In [59]:
#Gobernanza y etica de datos
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path
from pyspark.sql import SparkSession

print("=========Librerias importadas=========")

=========Librerias importadas=========


In [60]:
#Crear carpeta para Gobernanza

BASE_DIR = Path("/content/")
DATA_GOV = BASE_DIR / "datos" / "gobernanza"
DOCS_DIR = BASE_DIR / "documentos"

DATA_GOV.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos gobernanza: {DATA_GOV}")
print(f"Documentos: {DOCS_DIR}")


Datos gobernanza: /content/datos/gobernanza
Documentos: /content/documentos


In [61]:
#lectura dataset analitico de la guia 3
# inicio session en spark
spark = (
    SparkSession.builder
    .appName('DevOps_Metrics_Gobernanza04')
    .master('local[*]')
    .getOrCreate()
)

BASE_DIR = Path("/content/")
RUTA_PARQUET = BASE_DIR / 'datos' / 'analiticos' / 'devops_metrics_parquet_equipo'

# Leer el Parquet
df_analitico = spark.read.parquet(str(RUTA_PARQUET.resolve()))

# cantidad total de filas y de columnas
print('Filas totales en el dataset:', df_analitico.count())
print("\nColumnas disponibles para clasificar:")
print(df_analitico.columns)

df_analitico.limit(5).toPandas()

Filas totales en el dataset: 20144

Columnas disponibles para clasificar:
['company_id', 'product_area', 'event_date', 'ticket_count', 'deployment_count', 'failed_deployments', 'avg_lead_time_hours', 'rollback_count', 'incident_count', 'avg_resolution_time_hours', 'total_downtime_min', 'avg_cpu_usage_pct', 'avg_memory_usage_pct', 'avg_response_time_ms', 'avg_error_rate_pct', 'avg_availability_pct', 'avg_requests_per_minute', 'deployment_failure_rate_pct', 'rollback_rate_pct', 'anio', 'mes', 'dia', 'dia_semana', 'es_fin_semana', 'sin_datos_monitoreo']


,company_id,product_area,event_date,ticket_count,deployment_count,failed_deployments,avg_lead_time_hours,rollback_count,incident_count,avg_resolution_time_hours,...,avg_availability_pct,avg_requests_per_minute,deployment_failure_rate_pct,rollback_rate_pct,anio,mes,dia,dia_semana,es_fin_semana,sin_datos_monitoreo
0,100001,Analytics,2026-01-01,3,0,0,NaN,0,0,NaN,...,98.422,7051.0,NaN,NaN,2026,1,1,Thursday,False,False
1,100001,Analytics,2026-01-02,1,0,0,NaN,0,0,NaN,...,NaN,NaN,NaN,NaN,2026,1,2,Friday,False,True
2,100001,Analytics,2026-01-03,2,1,0,18.29,0,0,NaN,...,96.012,4186.0,0.0,0.0,2026,1,3,Saturday,True,False
3,100001,Analytics,2026-01-04,1,0,0,NaN,0,0,NaN,...,95.885,854.0,NaN,NaN,2026,1,4,Sunday,True,False
4,100001,Analytics,2026-01-05,1,0,0,NaN,0,0,NaN,...,NaN,NaN,NaN,NaN,2026,1,5,Monday,False,True


In [62]:
#CLASIFICACION DE DATOS

#NIVELES: PUBLICO, INTERNO, CONFIDENCIAL, RESTRINGIDO

clasificacion = pd.DataFrame([
    ["company_id", "Identificador interno", "Confidencial", "Vincular registros", "seudonimizar"],
    ["avg_error_rate_pct", "Métrica de calidad", "Confidencial", "Rendimiento", "Revisar granularidad"],
    ["total_downtime_min", "Métrica crítica", "Confidencial", "Estabilidad", "Excluir de vista analítica"],
    ["product_area", "Dato de negocio", "Interno", "Agrupación analítica", "Conservar"],
    ["event_date", "Dato temporal", "Interno", "Análisis de tendencias", "Generalizar por mes"],
    ["avg_resolution_time_hours", "Métrica operativa", "Interno", "Eficiencia de soporte", "Conservar"],
    ["deployment_failure_rate_pct", "Métrica de entrega", "Interno", "Calidad de CI/CD", "Conservar"],
    ["ticket_count", "Métrica operativa", "Interno", "Volumen de trabajo", "Conservar"],
    ["incident_count", "Métrica operativa", "Interno", "Volumen de incidentes", "Conservar"],
    ["avg_cpu_usage_pct", "Métrica de infraestructura", "Interno", "Rendimiento de hardware", "Conservar"],
    ["avg_availability_pct", "Métrica de negocio", "Público", "SLA de servicio", "Conservar"],
    ["rollback_count", "Métrica de entrega", "Interno", "Reversiones de sistema", "Conservar"]
],
columns=["campo", "tipo", "clasificacion", "finalidad", "control"])

print("Matriz de clasificación ---------------------")
print(clasificacion)

Matriz de clasificación ---------------------
                          campo                        tipo clasificacion  \
0                    company_id       Identificador interno  Confidencial   
1            avg_error_rate_pct          Métrica de calidad  Confidencial   
2            total_downtime_min             Métrica crítica  Confidencial   
3                  product_area             Dato de negocio       Interno   
4                    event_date               Dato temporal       Interno   
5     avg_resolution_time_hours           Métrica operativa       Interno   
6   deployment_failure_rate_pct          Métrica de entrega       Interno   
7                  ticket_count           Métrica operativa       Interno   
8                incident_count           Métrica operativa       Interno   
9             avg_cpu_usage_pct  Métrica de infraestructura       Interno   
10         avg_availability_pct          Métrica de negocio       Público   
11               rollback_coun

In [63]:
#Guardar clasificacion
ruta_clasificacion = DOCS_DIR / "clasificacion_datos_DevOps_Metrics_Analytics_ProyctoFinal04.xlsx"

clasificacion.to_excel(ruta_clasificacion, index=False)

print(f"Clasificación guardada {ruta_clasificacion}")

Clasificación guardada /content/documentos/clasificacion_datos_DevOps_Metrics_Analytics_ProyctoFinal04.xlsx


In [64]:
#SEUDONIMIZACION DE company_id
df_analitico_pd = df_analitico.toPandas()


SALT = "DevOps_Metrics_Analytics_ProyctoFinal04"

def tokenizar(valor):
    """Convierte un valor en un código único de 12 caracteres."""
    texto = f"{SALT}|{valor}".encode('utf-8')
    return hashlib.sha256(texto).hexdigest()[:12]

# Aplicamos la función sobre el DataFrame de Pandas
df_analitico_pd["company_token"] = df_analitico_pd["company_id"].astype(str).apply(tokenizar)

print("Seudonimización aplicada: -----------------------")
print(df_analitico_pd[["company_id", "company_token"]].head(10))

Seudonimización aplicada: -----------------------
   company_id company_token
0      100001  aaf240cdc2c0
1      100001  aaf240cdc2c0
2      100001  aaf240cdc2c0
3      100001  aaf240cdc2c0
4      100001  aaf240cdc2c0
5      100001  aaf240cdc2c0
6      100001  aaf240cdc2c0
7      100001  aaf240cdc2c0
8      100001  aaf240cdc2c0
9      100001  aaf240cdc2c0


In [65]:
# Revisar granularidad de avg_error_rate_pct
df_analitico_pd['avg_error_rate_pct_gen'] = df_analitico_pd['avg_error_rate_pct'].round(0)
print("Se redondendearon los decimales de avg_error_rate_pct_gen ")
print(df_analitico_pd[['avg_error_rate_pct', 'avg_error_rate_pct_gen']].head(5))

Se redondendearon los decimales de avg_error_rate_pct_gen 
   avg_error_rate_pct  avg_error_rate_pct_gen
0                7.24                     7.0
1                 NaN                     NaN
2                3.00                     3.0
3                5.59                     6.0
4                 NaN                     NaN


In [66]:
#GENERALIZACIÓN a event_date
# fecha en formato Año-Mes
df_analitico_pd['periodo_mes'] = pd.to_datetime(df_analitico_pd['event_date']).dt.strftime('%Y-%m')

print("Se generalizo event_date")
print(" ANTES (event_date) ,  DESPUÉS (periodo_mes):")
print(df_analitico_pd[['event_date', 'periodo_mes']].head(5))
print("\n")

# eliminamos la columna anterior
df_analitico_pd = df_analitico_pd.drop(columns=['event_date'])

Se generalizo event_date
 ANTES (event_date) ,  DESPUÉS (periodo_mes):
   event_date periodo_mes
0  2026-01-01     2026-01
1  2026-01-02     2026-01
2  2026-01-03     2026-01
3  2026-01-04     2026-01
4  2026-01-05     2026-01




In [67]:
# PRINCIPIO DE MINIMIZACION de total_downtime_min
vista_analitica = df_analitico_pd[[
    "company_token",
    "avg_error_rate_pct_gen",
    "product_area",
    "periodo_mes",
    "avg_resolution_time_hours",
    "deployment_failure_rate_pct",
    "ticket_count",
    "incident_count",
    "avg_cpu_usage_pct",
    "avg_availability_pct",
    "rollback_count"
]].copy()

print("VISTA ANALITICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)")
print(vista_analitica.head(5))

VISTA ANALITICA PROTEGIDA (SIN IDENTIFICADORES DIRECTOS)
  company_token  avg_error_rate_pct_gen product_area periodo_mes  \
0  aaf240cdc2c0                     7.0    Analytics     2026-01   
1  aaf240cdc2c0                     NaN    Analytics     2026-01   
2  aaf240cdc2c0                     3.0    Analytics     2026-01   
3  aaf240cdc2c0                     6.0    Analytics     2026-01   
4  aaf240cdc2c0                     NaN    Analytics     2026-01   

   avg_resolution_time_hours  deployment_failure_rate_pct  ticket_count  \
0                        NaN                          NaN             3   
1                        NaN                          NaN             1   
2                        NaN                          0.0             2   
3                        NaN                          NaN             1   
4                        NaN                          NaN             1   

   incident_count  avg_cpu_usage_pct  avg_availability_pct  rollback_count  
0     

In [68]:
#VISTA PROTEGIDA

ruta_publicable = DATA_GOV/"dataset_publicable_DevOps_Metrics_Analytics_ProyectoFinal04.csv"

vista_analitica.to_csv(ruta_publicable, index=False, encoding="utf-8")

print(f"Vista protegida guardada: {ruta_publicable}")

Vista protegida guardada: /content/datos/gobernanza/dataset_publicable_DevOps_Metrics_Analytics_ProyectoFinal04.csv


In [69]:
#VALIDACION AUTOMÁTICA

# Definimos las columnas originales que NO deben llegar a la vista analítica
identificadores_directos = {"company_id", "total_downtime_min", "event_date"}
expuestos = identificadores_directos.intersection(vista_analitica.columns)

print("Verificando identificadores directos: -------------")
print(f"Indentificadores directos expuestos: {expuestos}")

if len(expuestos) == 0:
    print("Ningún identificador directo expuesto!!!!!")
else:
    print("ALERTA: Identificadores directos expuestos!!!!!!!!")

Verificando identificadores directos: -------------
Indentificadores directos expuestos: set()
Ningún identificador directo expuesto!!!!!


In [70]:
# MATRIZ DE RIESGOS
# Escala: Probabilidad e Impacto del 1 al 5
# Puntaje = Probabilidad x Impacto

riesgos = pd.DataFrame([
    ["Reidentificación de cliente original (company_id)", 3, 5, "Seudonimizar ", "Ingeniero de Datos", "Script de seudonimización"],
    ["Fuga de métrica de calidad confidencial (avg_error_rate_pct)", 3, 4, "Revisar granularidad", "Analista de Datos", "Script de ajuste"],
    ["Exposición de métrica crítica (total_downtime_min)", 4, 5, "Excluir de vista analítica (Minimización)", "Ingeniero de Datos", "Script de minimización"],
    ["Reidentificación por fecha exacta de evento (event_date)", 3, 4, "Generalizar por mes", "Ingeniero de Datos", "Script de generalización"]
],
columns=["riesgo", "probabilidad", "impacto", "control", "responsable", "evidencia"])

# Cálculos de criticidad
riesgos["puntaje"] = riesgos["probabilidad"] * riesgos["impacto"]
riesgos["nivel"] = riesgos["puntaje"].apply(
    lambda x: "Crítico" if x >= 16 else "Alto" if x >= 10 else "Moderado" if x >= 5 else "Bajo"
)

print("MATRIZ DE RIESGOS DEVOPS:")
print(riesgos.sort_values("puntaje", ascending=False))

MATRIZ DE RIESGOS DEVOPS:
                                              riesgo  probabilidad  impacto  \
2  Exposición de métrica crítica (total_downtime_...             4        5   
0  Reidentificación de cliente original (company_id)             3        5   
1  Fuga de métrica de calidad confidencial (avg_e...             3        4   
3  Reidentificación por fecha exacta de evento (e...             3        4   

                                     control         responsable  \
2  Excluir de vista analítica (Minimización)  Ingeniero de Datos   
0                              Seudonimizar   Ingeniero de Datos   
1                       Revisar granularidad   Analista de Datos   
3                        Generalizar por mes  Ingeniero de Datos   

                   evidencia  puntaje    nivel  
2     Script de minimización       20  Crítico  
0  Script de seudonimización       15     Alto  
1           Script de ajuste       12     Alto  
3   Script de generalización       12    

In [72]:
#GUARDAR LA MATRIZ DE RIESGOS

ruta_riesgos = DOCS_DIR/"Matriz_Riesgos_DevOps_Metrics_Analytics_ProyectoFinal04.xlsx" #
riesgos.to_excel(ruta_riesgos, index=False)

print(f"Matriz de riesgos guardada ------------- {ruta_riesgos}")

Matriz de riesgos guardada ------------- /content/documentos/Matriz_Riesgos_DevOps_Metrics_Analytics_ProyectoFinal04.xlsx


In [73]:
# POLÍTICA BREVE DE GOBERNANZA - Reglas de uso, acceso y protección #

politica = """
# Política de Gobernanza - Proyecto DevOps Metrics

## 1. Propósito
Definir cómo se utilizan, protegen y comparten los datos del proyecto , con el objetivo de crear
metricas y dashboard seguros para un buen analisis

## 2. Clasificación
- Público: puede compartirse sin restricciones como la Columna avg_availability_pct
- Interno: uso dentro del equipo
- Confidencial: requiere protección (company_id, total_downtime_min)
- Restringido: acceso limitado, ej: el Parquet original

## 3. Acceso por Rol
- Líder: Tiene acceso total para la toma de decisiones y definición de la política de datos.
- Ingeniero de Datos: Acceso a los datos crudos originales para aplicar scripts de limpieza y minimización.
- Analista de Datos y Calidad: Solo vista protegida (sin identificadores) para crear reportes y validar que no haya fugas de información.
- Docente: Solo lectura de evidencias (Matriz de Riesgos y CSV final) para evaluación y calificación del proyecto.

## 4. Minimización
* Se aplicó una técnica de selección estricta ("lista blanca") en el código para excluir por completo la
métrica confidencial de inactividad (total_downtime_min) de la vista analítica.

## 5. Retención
Los datos se conservan durante el ciclo IV 2026 y se eliminan al finalizar dicho ciclo el 20 de noviembre de 2026.

## 6. Ética
No se utilizan variables sin finalidad justificada ni se presentan conclusiones sin contexto.
"""



In [74]:
# GUARDAR POLÍTICA

ruta_politica = DOCS_DIR / "politica_gobernanza_DevOps_Metrics_Analytics_ProyectoFinal04.md" #
with open(ruta_politica, "w", encoding="utf-8") as f:
    f.write(politica)

print(f"Política guardada: {ruta_politica}")

Política guardada: /content/documentos/politica_gobernanza_DevOps_Metrics_Analytics_ProyectoFinal04.md
